# Lab 02 — Giới thiệu Deep Learning Frameworks: Thực hành với PyTorch

**Nội dung**: Tensor · Hardware Acceleration · Autograd · Xây dựng & huấn luyện Image Classifier (Fashion MNIST)

**Bài giảng nguồn**: 13 – Introduction of Deep Learning Frameworks  
**Thời lượng đề xuất**: 150–180 phút  
**Môi trường**: Google Colab (khuyến nghị, có GPU miễn phí) hoặc máy cá nhân có Python + PyTorch

---

## Phần 0 — Khởi động môi trường (~15 phút)

**Mục tiêu**: Dựng môi trường và ghi lại "dấu vân tay" máy của bạn.

### 0.1 – Kiểm tra PyTorch & thiết bị

**THỰC HÀNH**: Trên Colab: vào Runtime → Change runtime type → chọn GPU (T4). Sau đó chạy:

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
print("Device name    :", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU only")

### GIẢI THÍCH 0.1

**Câu hỏi**: Theo slide, một framework "làm phần toán cấp thấp để bạn tập trung vào kiến trúc". Hãy kể ra HAI việc cấp thấp mà framework làm hộ bạn.

**Trả lời**:

1. **Tính đạo hàm tự động (Automatic Differentiation / Autograd)**: Thay vì phải tự viết công thức đạo hàm cho từng phép toán và implement backpropagation bằng tay, framework tự dựng computation graph và tính gradient cho mọi tham số chỉ bằng một lệnh `.backward()`.

2. **Quản lý bộ nhớ và thực thi trên phần cứng (GPU/CPU memory management & kernel dispatch)**: Framework tự động phân bổ bộ nhớ trên GPU, chuyển dữ liệu giữa CPU↔GPU, gọi các CUDA kernel tối ưu cho phép nhân ma trận, convolution, v.v. — ta chỉ cần `.to(device)` mà không cần viết code CUDA.

---
## Phần 1 — Tensor: cấu trúc dữ liệu lõi (~25 phút)

Tensor là một mảng nhiều chiều có shape và dtype. Giống NumPy array, nhưng có thêm hai "siêu năng lực": chạy được trên GPU, và hỗ trợ auto-differentiation.

### 1.1 – Tạo tensor & đọc thuộc tính

In [ ]:
import torch
X = torch.tensor([[1.0, 4.0, 7.0],
                  [2.0, 3.0, 6.0]])
print(X.shape)   # ?
print(X.dtype)   # ?

### DỰ ĐOÁN 1.1

**Câu hỏi**: Trước khi chạy, dự đoán `X.shape` và `X.dtype`.

**Trả lời**:
- `X.shape` = `torch.Size([2, 3])` — vì X có 2 hàng, 3 cột.
- `X.dtype` = `torch.float32` — vì các số có dấu chấm thập phân (1.0, 4.0, ...) nên PyTorch mặc định dùng float32.

### 1.2 – Phép toán giống NumPy & nhân ma trận

In [ ]:
print(X.mean())     # trung bình toàn bộ phần tử
print(X @ X.T)      # nhân ma trận X (2x3) với X^T (3x2)

### DỰ ĐOÁN 1.2 (tính tay)

**Câu hỏi**: Tính TAY tích X @ X.T. Kết quả là ma trận 2×2. Điền 4 số.

**Trả lời**:

X = [[1, 4, 7], [2, 3, 6]], X.T = [[1, 2], [4, 3], [7, 6]]

- `[0][0]` = 1×1 + 4×4 + 7×7 = 1 + 16 + 49 = **66**
- `[0][1]` = 1×2 + 4×3 + 7×6 = 2 + 12 + 42 = **56**
- `[1][0]` = 2×1 + 3×4 + 6×7 = 2 + 12 + 42 = **56**
- `[1][1]` = 2×2 + 3×3 + 6×6 = 4 + 9 + 36 = **49**

Kết quả: `[[66, 56], [56, 49]]`

**Vì sao X @ X.T cho ma trận 2×2?** Vì quy tắc nhân (m×n)·(n×p) → (m×p). Ở đây X là (2×3), X.T là (3×2), nên kết quả là (2×2). Nếu nhân ngược X.T @ X sẽ cho (3×3).

### 1.3 – In-place ops & cầu nối NumPy

In [ ]:
import numpy as np
arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(arr)   # KHÔNG sao chép — dùng chung bộ nhớ
arr[0] = 99.0               # sửa mảng NumPy...
print(t)                    # ...tensor có đổi theo không?
print(t.dtype)

### DỰ ĐOÁN 1.3

**Câu hỏi**: Sau khi gán `arr[0] = 99`, giá trị `t` in ra là gì? `t.dtype` là float32 hay float64?

**Trả lời**:
- `t` = `tensor([99., 2., 3.])` — vì `torch.from_numpy` **dùng chung bộ nhớ** (shared memory), không sao chép. Khi sửa `arr`, `t` cũng thay đổi theo.
- `t.dtype` = `torch.float64` — vì NumPy mặc định tạo mảng `float64`, và `from_numpy` giữ nguyên dtype gốc.

**Mẹo**: Các op kết thúc bằng dấu gạch dưới (in-place) sửa trực tiếp tensor để tiết kiệm bộ nhớ: `X.zero_()`, `X.relu_()`. Thử `X.zero_()` rồi in X:

In [ ]:
X.zero_()
print(X)

### GIẢI THÍCH 1.4

**Câu hỏi**: Vì sao `torch.from_numpy` lại "nguy hiểm" nếu bạn vô tình sửa mảng NumPy sau đó? Nêu một tình huống bug.

**Trả lời**:

Vì `from_numpy` dùng chung bộ nhớ (zero-copy), bất kỳ thay đổi nào trên mảng NumPy gốc sẽ **âm thầm** thay đổi tensor mà ta không hay biết. 

**Tình huống bug**: Trong một pipeline xử lý dữ liệu, giả sử ta load ảnh vào `arr`, tạo tensor `t = torch.from_numpy(arr)`, rồi ở bước tiền xử lý tiếp theo ta normalize `arr` (ví dụ `arr /= 255.0`). Lúc này tensor `t` cũng bị chia 255 theo — nhưng nếu ta tưởng `t` vẫn giữ giá trị pixel gốc (0–255) để dùng ở nơi khác, model sẽ nhận input sai mà không có lỗi nào được raise. Cách an toàn: dùng `torch.tensor(arr)` hoặc `torch.from_numpy(arr).clone()` để tạo bản sao độc lập.

---
## Phần 2 — Hardware Acceleration (~25 phút)

GPU có hàng nghìn nhân chạy song song; deep learning = rất nhiều phép nhân ma trận lớn.

### 2.1 – Chọn thiết bị

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"   # chip Apple M1/M2/M3
else:
    device = "cpu"
print("Using:", device)

### 2.2 – Đo tốc độ: CPU vs GPU

**THỰC HÀNH**: Nhân ma trận 1000×1000, lặp 50 lần. Lưu ý warm-up + synchronize để đo đúng.

In [ ]:
import time
N = 1000
A = torch.randn(N, N)
B = torch.randn(N, N)

# --- CPU ---
t0 = time.time()
for _ in range(50):
    _ = A @ B
cpu_t = time.time() - t0

# --- GPU ---
Ag, Bg = A.to(device), B.to(device)
_ = Ag @ Bg                                  # warm-up
if device == "cuda": torch.cuda.synchronize()
t0 = time.time()
for _ in range(50):
    _ = Ag @ Bg
if device == "cuda": torch.cuda.synchronize()
gpu_t = time.time() - t0

print(f"CPU: {cpu_t:.4f}s | GPU: {gpu_t:.4f}s | speedup x{cpu_t/gpu_t:.1f}")

### DỰ ĐOÁN 2.2

**Câu hỏi**: Trước khi chạy, đoán GPU nhanh hơn CPU khoảng bao nhiêu lần?

**Trả lời**:
- Dự đoán: khoảng **20–50×** — slide cho ví dụ ~29×, và ma trận 1000×1000 đủ lớn để GPU phát huy ưu thế song song.
- Số đo thực tế: *(ghi sau khi chạy — thường rơi vào 10–50× tùy GPU)*

### GIẢI THÍCH 2.3

**Câu hỏi**: Vì sao GPU thắng CPU ở tác vụ này? Và vì sao nếu N = 10 thì GPU có thể KHÔNG nhanh hơn?

**Trả lời**:

GPU thắng vì phép nhân ma trận 1000×1000 bao gồm hàng triệu phép nhân-cộng độc lập nhau, có thể chạy song song trên hàng nghìn CUDA cores của GPU. CPU chỉ có vài chục cores nên phải xử lý tuần tự nhiều hơn.

Khi N = 10 (ma trận rất nhỏ, chỉ 100 phần tử), lượng tính toán quá ít — không đủ "lấp đầy" hàng nghìn core GPU. Lúc này, **chi phí overhead** chiếm ưu thế: thời gian copy dữ liệu từ CPU RAM sang GPU VRAM, thời gian khởi chạy kernel, và thời gian đồng bộ. Những chi phí này cố định (vài chục μs) và lớn hơn so với thời gian tính toán thực sự, khiến GPU có thể chậm hơn CPU.

### 2.4 – SĂN LỖI: thiết bị không khớp

**Code dưới đây sẽ báo lỗi runtime nếu máy có GPU.** Chạy nó, đọc lỗi, chẩn đoán và sửa.

In [ ]:
# SĂN LỖI — Code cố tình sai
if device != "cpu":
    a = torch.tensor([1.0, 2.0, 3.0]).to(device)  # tensor trên GPU
    b = torch.tensor([4.0, 5.0, 6.0])             # tensor trên CPU (quên .to(device))
    try:
        c = a + b  # LỖI: hai tensor ở khác device
    except RuntimeError as e:
        print("LỖI:", e)

### Chẩn đoán & Sửa lỗi 2.4

**Thông báo lỗi**: `Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!`

**Nguyên nhân**: Tensor `a` nằm trên GPU (`cuda:0`) nhưng tensor `b` vẫn ở CPU. PyTorch không tự động chuyển device — hai tensor phải ở cùng device mới thực hiện phép toán được.

**Cách sửa (1 dòng)**: Thêm `.to(device)` cho `b`:
```python
b = torch.tensor([4.0, 5.0, 6.0]).to(device)
```

In [ ]:
# Code đã sửa
if device != "cpu":
    a = torch.tensor([1.0, 2.0, 3.0]).to(device)
    b = torch.tensor([4.0, 5.0, 6.0]).to(device)  # ĐÃ SỬA: thêm .to(device)
    c = a + b
    print(c)

---
## Phần 3 — Autograd: đạo hàm tự động (~30 phút)

Autograd theo dõi mọi phép toán trên tensor có `requires_grad=True`, dựng computation graph động, rồi khi gọi `.backward()` sẽ lan truyền ngược (backpropagation) để tính gradient, lưu vào `.grad`.

### 3.1 – Đạo hàm bậc nhất, kiểm tra bằng tay

### DỰ ĐOÁN 3.1 (làm toán trước)

**Câu hỏi**: Cho f(x) = x². Tính TAY f'(x), rồi thế x = 5.

**Trả lời**:
- f'(x) = **2x**
- f'(5) = 2 × 5 = **10**

Giờ kiểm chứng bằng Autograd:

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
y = x ** 2
y.backward()       # lan truyền ngược
print(x.grad)      # so với f'(5) = 10 tính tay → khớp!

### 3.2 – Nhiều biến

### DỰ ĐOÁN 3.2

**Câu hỏi**: Cho f(a, b) = a²·b + b tại a=2, b=3. Tính tay đạo hàm riêng.

**Trả lời**:
- ∂f/∂a = 2a·b = 2×2×3 = **12**
- ∂f/∂b = a² + 1 = 4 + 1 = **5**

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
f = a**2 * b + b
f.backward()
print(a.grad, b.grad)  # Kỳ vọng: tensor(12.) tensor(5.) → khớp với tính tay!

### 3.3 – Vì sao phải zero_grad()? (thí nghiệm tích lũy gradient)

**THỰC HÀNH 3.3**: Chạy backward HAI lần liên tiếp mà không reset gradient:

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
(x**2).backward()
print("lần 1:", x.grad)
(x**2).backward()      # KHÔNG reset
print("lần 2:", x.grad)

### DỰ ĐOÁN 3.3

**Câu hỏi**: "lần 2" in ra số mấy? Vì sao? Liên hệ với `optimizer.zero_grad()`.

**Trả lời**:
- Lần 1: `x.grad` = **10** (= 2×5, đúng đạo hàm).
- Lần 2: `x.grad` = **20** (= 10 + 10). PyTorch **cộng dồn** gradient vào `.grad` thay vì ghi đè. Lần backward thứ 2 tính gradient = 10 rồi cộng thêm vào giá trị cũ 10 → thành 20.

**Liên hệ**: Trong training loop, ta luôn gọi `optimizer.zero_grad()` trước mỗi batch để reset gradient về 0. Nếu quên, gradient sẽ tích lũy qua các batch → model cập nhật sai hướng, loss dao động hoặc phân kỳ.

### 3.4 – no_grad() khi suy luận

In [ ]:
with torch.no_grad():
    z = x ** 2
print(z.requires_grad)   # True hay False?

### GIẢI THÍCH 3.4

**Câu hỏi**: Khi suy luận (inference), ta bọc code trong `torch.no_grad()`. Việc này giúp tiết kiệm gì? Vì sao TRAIN thì không dùng?

**Trả lời**:

`z.requires_grad` = **False**. Trong khối `no_grad()`, PyTorch **không xây dựng computation graph** và **không lưu trữ các tensor trung gian** cần cho backpropagation. Điều này:
- **Tiết kiệm bộ nhớ**: không phải lưu các activation trung gian (có thể chiếm hàng GB với mạng lớn).
- **Tăng tốc**: bỏ qua overhead của việc theo dõi phép toán.

Khi TRAIN thì ta **cần** computation graph để gọi `.backward()` tính gradient → cập nhật trọng số. Nếu dùng `no_grad()` lúc train, gradient = None và model không học được gì.

---
## Phần 4 — Xây dựng & huấn luyện Image Classifier (~50 phút)

**Mục tiêu**: Ghép mọi mảnh ghép thành một MLP phân loại ảnh trên Fashion MNIST (10 lớp quần áo).  
**Quy trình**: Load → Explore → Build → Train → Evaluate.

### 4.1 – Load dữ liệu với TorchVision

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform = transforms.ToTensor()   # scale pixel về [0,1], shape [1,28,28]

train_full = datasets.FashionMNIST(root="data", train=True,
                                   download=True, transform=transform)
test_set   = datasets.FashionMNIST(root="data", train=False,
                                   download=True, transform=transform)

train_set, val_set = random_split(train_full, [55000, 5000])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False)

### GIẢI THÍCH 4.1

**Câu hỏi**: Vì sao `shuffle=True` cho tập train nhưng `shuffle=False` cho val/test?

**Trả lời**:

- **Train `shuffle=True`**: Trộn dữ liệu giúp mỗi epoch model thấy thứ tự mẫu khác nhau, tránh học theo thứ tự (ordering bias). Nếu không trộn và dữ liệu được sắp theo lớp (tất cả T-shirt rồi đến Trouser...), model sẽ chỉ học lớp cuối và quên lớp đầu (catastrophic forgetting theo batch).

- **Val/Test `shuffle=False`**: Khi đánh giá, ta chỉ cần tính accuracy — thứ tự không ảnh hưởng kết quả. Giữ thứ tự cố định giúp kết quả reproducible (chạy lại cho cùng số), và debug dễ hơn (biết ảnh nào model dự đoán sai).

### 4.2 – Explore: hiểu shape của ảnh

In [ ]:
X_sample, y_sample = train_set[0]
print(X_sample.shape)    # ?  (quy ước channel-first của PyTorch)
print(y_sample)          # nhãn là số nguyên 0..9

classes = ["T-shirt/top","Trouser","Pullover","Dress","Coat",
           "Sandal","Shirt","Sneaker","Bag","Ankle boot"]

import matplotlib.pyplot as plt
plt.imshow(X_sample.squeeze(), cmap="gray")  # squeeze bỏ chiều channel để vẽ
plt.title(classes[y_sample]); plt.show()

### DỰ ĐOÁN 4.2

**Câu hỏi**: `X_sample.shape` = ? Số đầu (channel) bằng 1 — vì sao? Ảnh RGB thì số đó là mấy? Vì sao phải `.squeeze()` cho Matplotlib?

**Trả lời**:
- `X_sample.shape` = **[1, 28, 28]** (channel=1, height=28, width=28).
- Channel = 1 vì Fashion MNIST là ảnh **xám** (grayscale), chỉ có 1 kênh màu. Ảnh RGB sẽ có **3** kênh (Red, Green, Blue).
- Phải `.squeeze()` vì PyTorch dùng quy ước **channel-first** `[C, H, W]` nhưng Matplotlib mong đợi ảnh 2D `[H, W]` cho ảnh xám. `.squeeze()` loại bỏ chiều C=1 để shape thành `[28, 28]`.

### 4.3 – Build: định nghĩa ImageClassifier (MLP 2 hidden layers)

In [ ]:
import torch.nn as nn

class ImageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),            # [B,1,28,28] -> [B,784]
            nn.Linear(28*28, 300),
            nn.ReLU(),
            nn.Linear(300, 100),
            nn.ReLU(),
            nn.Linear(100, 10),      # 10 logits, KHÔNG activation ở output
        )
    def forward(self, x):
        return self.net(x)

model = ImageClassifier().to(device)

### DỰ ĐOÁN 4.3 (truy vết shape)

**Câu hỏi**: Một batch đi qua model. Điền shape sau mỗi tầng (B = 32).

**Trả lời**:

| Tầng | Output shape |
|---|---|
| Input | [32, 1, 28, 28] |
| Flatten | [32, 784] |
| Linear(784→300) | [32, 300] |
| ReLU | [32, 300] |
| Linear(300→100) | [32, 100] |
| ReLU | [32, 100] |
| Linear(100→10) | [32, 10] |

**(a) Vì sao tầng Linear đầu phải có đúng 784 input?** Vì Flatten biến ảnh [1, 28, 28] thành vector 1×28×28 = 784 phần tử. Nếu input features ≠ 784, PyTorch sẽ báo shape mismatch.

**(b) Vì sao tầng cuối có đúng 10 output?** Vì Fashion MNIST có 10 lớp (T-shirt, Trouser, ..., Ankle boot). Mỗi output là logit cho 1 lớp.

In [ ]:
# Kiểm chứng shape
print(model(next(iter(train_loader))[0].to(device)).shape)

### 4.4 – Train: loss function & training loop

> **Vì sao `nn.CrossEntropyLoss`?** CrossEntropyLoss đã tự áp log-softmax bên trong, tính trực tiếp từ raw logits — ổn định số học hơn và nhanh hơn. Vì thế output model để nguyên logits; chỉ gọi `F.softmax()` khi cần xác suất.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(10):
    model.train()
    running = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        logits = model(Xb)
        loss   = criterion(logits, yb)
        optimizer.zero_grad()    # <- nhớ Phần 3.3!
        loss.backward()
        optimizer.step()
        running += loss.item()
    print(f"Epoch {epoch+1}: loss = {running/len(train_loader):.4f}")

### THỰC HÀNH 4.4

**Ghi lại loss**: Loss giảm dần qua các epoch, cho thấy model đang **học** — tức là trọng số đang được cập nhật theo hướng giảm hàm mất mát. Thường thấy:
- Epoch 1 loss ≈ 1.5–2.0 (model chưa học gì, gần như random → -log(1/10) ≈ 2.3)
- Epoch 5 loss ≈ 0.5–0.7
- Epoch 10 loss ≈ 0.4–0.5

### 4.5 – Evaluate: dự đoán & tính accuracy

In [ ]:
import torch.nn.functional as F
model.eval()
correct = total = 0
with torch.no_grad():                      # <- Phần 3.4
    for Xb, yb in val_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        logits = model(Xb)
        preds  = logits.argmax(dim=1)      # lớp có logit cao nhất
        correct += (preds == yb).sum().item()
        total   += yb.size(0)
print(f"Validation accuracy = {correct/total:.4f}")

# Xác suất + top-4 cho 1 batch:
Xb, yb = next(iter(val_loader))
with torch.no_grad():
    probs = F.softmax(model(Xb.to(device)), dim=1)
topv, topi = torch.topk(probs, k=4, dim=1)

### THỰC HÀNH 4.5

**Validation accuracy** sau 10 epoch thường rơi vào khoảng **0.82–0.86**.

### GIẢI THÍCH 4.5

**Câu hỏi**: Tại sao ta dùng `argmax(logits)` để lấy nhãn dự đoán mà KHÔNG cần softmax?

**Trả lời**: Vì softmax là hàm **đơn điệu tăng** — nó không làm thay đổi thứ tự lớn-nhỏ giữa các phần tử. Phần tử nào có logit cao nhất thì sau softmax vẫn có xác suất cao nhất. Do đó `argmax(logits)` cho cùng kết quả với `argmax(softmax(logits))`. Chỉ khi cần **giá trị xác suất** (ví dụ hiển thị % cho user) ta mới cần softmax.

### 4.6 – BA cuộc SĂN LỖI

#### SĂN LỖI A — softmax trước CrossEntropyLoss

In [ ]:
# Một bạn sửa training loop thành thế này:
# logits = F.softmax(model(Xb), dim=1)  # ← THÊM softmax
# loss = criterion(logits, yb)
# → Model học chậm/kém hẳn!

# Minh họa sự khác biệt:
model_test = ImageClassifier().to(device)
Xb_test, yb_test = next(iter(train_loader))
Xb_test, yb_test = Xb_test.to(device), yb_test.to(device)

logits_raw = model_test(Xb_test)
logits_softmaxed = F.softmax(model_test(Xb_test), dim=1)

loss_correct = criterion(logits_raw, yb_test)
loss_wrong = criterion(logits_softmaxed, yb_test)
print(f"Loss (đúng, raw logits):  {loss_correct.item():.4f}")
print(f"Loss (sai, softmax trước): {loss_wrong.item():.4f}")

**Chẩn đoán**: `CrossEntropyLoss` đã tự áp softmax (cụ thể là log-softmax) bên trong. Nếu ta softmax trước rồi đưa vào `CrossEntropyLoss`, thì thành **softmax HAI lần**: softmax(softmax(logits)). Kết quả là các xác suất bị "nén" về gần nhau, gradient rất nhỏ → model học cực chậm.

**Cách sửa**: Bỏ `F.softmax()`, đưa raw logits trực tiếp vào `criterion()`.

#### SĂN LỖI B — sai số input tầng đầu

In [ ]:
# Code lỗi: đổi nn.Linear(28*28, 300) thành nn.Linear(28, 300)
class BuggyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28, 300),   # BUG: 28 thay vì 784
            nn.ReLU(),
            nn.Linear(300, 100),
            nn.ReLU(),
            nn.Linear(100, 10),
        )
    def forward(self, x):
        return self.net(x)

buggy = BuggyClassifier().to(device)
try:
    buggy(Xb_test)
except RuntimeError as e:
    print("LỖI:", e)

**Thông báo lỗi**: `RuntimeError: mat1 and mat2 shapes cannot be multiplied (32×784 and 28×300)`

**Chẩn đoán**: 
- Con số **784** đến từ Flatten: ảnh [1, 28, 28] được duỗi thành vector 784 phần tử.
- Con số **28** là input features ta khai báo sai cho Linear layer.
- Nhân ma trận (32×**784**) @ (**28**×300) yêu cầu chiều trong phải khớp (784 ≠ 28) → lỗi.

**Cách sửa**: Đổi lại `nn.Linear(28*28, 300)` hoặc `nn.Linear(784, 300)` để khớp với output của Flatten.

#### SĂN LỖI C — quên zero_grad()

In [ ]:
# Huấn luyện KHÔNG có zero_grad()
model_c = ImageClassifier().to(device)
optimizer_c = torch.optim.SGD(model_c.parameters(), lr=0.01)
criterion_c = nn.CrossEntropyLoss()

for epoch in range(5):
    model_c.train()
    running = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        logits = model_c(Xb)
        loss = criterion_c(logits, yb)
        # optimizer_c.zero_grad()  # ← ĐÃ XÓA (BUG)
        loss.backward()
        optimizer_c.step()
        running += loss.item()
    print(f"Epoch {epoch+1} (NO zero_grad): loss = {running/len(train_loader):.4f}")

**Quan sát**: Loss không hội tụ "bình thường" — có thể dao động mạnh hoặc không giảm ổn định.

**Giải thích**: Liên hệ với Phần 3.3, khi không gọi `zero_grad()`, gradient bị **cộng dồn** qua mọi batch. Sau 100 batch, `.grad` chứa tổng gradient của cả 100 batch thay vì chỉ batch hiện tại. Bước cập nhật `optimizer.step()` dùng gradient tích lũy khổng lồ này → trọng số "nhảy" quá xa → loss dao động, model không học ổn định.

**Cách sửa**: Thêm lại `optimizer.zero_grad()` trước `loss.backward()` trong mỗi iteration.

---
## Phần 5 — Thí nghiệm & tổng hợp (~25 phút)

Mỗi thí nghiệm: đổi MỘT thứ, giữ nguyên phần còn lại, huấn luyện lại, ghi accuracy.

In [ ]:
# Hàm tiện ích: train và evaluate
def train_and_eval(model, train_loader, val_loader, lr=0.01, epochs=10):
    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            loss = criterion(model(Xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item()
        train_loss = running / len(train_loader)
    
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            correct += (model(Xb).argmax(1) == yb).sum().item()
            total += yb.size(0)
    val_acc = correct / total
    print(f"  → Val acc = {val_acc:.4f}, Train loss (last) = {train_loss:.4f}")
    return val_acc, train_loss

### Thí nghiệm 1: Baseline (lr=0.01, hidden 300→100, 10 epoch)

In [ ]:
print("=== Baseline ===")
baseline = ImageClassifier()
acc_base, loss_base = train_and_eval(baseline, train_loader, val_loader, lr=0.01, epochs=10)

### Thí nghiệm 2: lr = 0.001 (nhỏ hơn 10×)

In [ ]:
print("=== lr = 0.001 ===")
m2 = ImageClassifier()
acc_lr001, loss_lr001 = train_and_eval(m2, train_loader, val_loader, lr=0.001, epochs=10)

### Thí nghiệm 3: lr = 1.0 (lớn)

In [ ]:
print("=== lr = 1.0 ===")
m3 = ImageClassifier()
acc_lr1, loss_lr1 = train_and_eval(m3, train_loader, val_loader, lr=1.0, epochs=10)

### Thí nghiệm 4: Bỏ cả hai nn.ReLU() (model tuyến tính)

In [ ]:
class LinearClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 300),
            # KHÔNG có ReLU
            nn.Linear(300, 100),
            # KHÔNG có ReLU
            nn.Linear(100, 10),
        )
    def forward(self, x):
        return self.net(x)

print("=== No ReLU ===")
m4 = LinearClassifier()
acc_norelu, loss_norelu = train_and_eval(m4, train_loader, val_loader, lr=0.01, epochs=10)

### Thí nghiệm 5: Hidden nhỏ (32→16 thay vì 300→100)

In [ ]:
class SmallClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 10),
        )
    def forward(self, x):
        return self.net(x)

print("=== Small hidden ===")
m5 = SmallClassifier()
acc_small, loss_small = train_and_eval(m5, train_loader, val_loader, lr=0.01, epochs=10)

### Thí nghiệm 6: Train 20 epoch thay vì 10

In [ ]:
print("=== 20 epochs ===")
m6 = ImageClassifier()
acc_20ep, loss_20ep = train_and_eval(m6, train_loader, val_loader, lr=0.01, epochs=20)

### Bảng tổng hợp kết quả

In [ ]:
print("\n" + "="*60)
print(f"{'Thí nghiệm':<45} {'Val Acc':>8} {'Loss':>8}")
print("="*60)
print(f"{'Baseline (lr=0.01, 300→100, 10ep)':<45} {acc_base:>8.4f} {loss_base:>8.4f}")
print(f"{'lr = 0.001':<45} {acc_lr001:>8.4f} {loss_lr001:>8.4f}")
print(f"{'lr = 1.0':<45} {acc_lr1:>8.4f} {loss_lr1:>8.4f}")
print(f"{'No ReLU (linear)':<45} {acc_norelu:>8.4f} {loss_norelu:>8.4f}")
print(f"{'Small hidden (32→16)':<45} {acc_small:>8.4f} {loss_small:>8.4f}")
print(f"{'20 epochs':<45} {acc_20ep:>8.4f} {loss_20ep:>8.4f}")
print("="*60)

### GIẢI THÍCH 5.1 — Learning rate

**So sánh lr=0.001, 0.01, 1.0 từ bảng:**

- **lr=0.001** (quá nhỏ): Model học rất chậm — sau 10 epoch loss vẫn cao, accuracy thấp hơn baseline. Gradient bước quá nhỏ, cần nhiều epoch hơn mới hội tụ.

- **lr=0.01** (baseline): Cân bằng tốt — loss giảm ổn định, accuracy đạt ~0.83–0.86.

- **lr=1.0** (quá lớn): Loss có thể "nổ" (NaN) hoặc dao động mạnh, accuracy rất thấp hoặc không học được gì. Gradient bước quá lớn khiến trọng số "nhảy" qua đáy hàm loss, không bao giờ hội tụ.

### GIẢI THÍCH 5.2 — Vì sao cần ReLU?

**Khi bỏ hết ReLU**, accuracy giảm rõ rệt (thường ~0.82–0.84 → ~0.78–0.82). Lý do: một chuỗi nhiều tầng Linear **không có hàm phi tuyến** thì về bản chất tương đương **MỘT tầng Linear duy nhất**. Chứng minh: nếu y = W₃(W₂(W₁·x)), thì y = (W₃·W₂·W₁)·x = W'·x — chỉ là một phép biến đổi tuyến tính. Ba tầng nhưng "sức mạnh" chỉ bằng một tầng!

ReLU (và các activation phi tuyến khác) **phá vỡ tính tuyến tính** giữa các tầng, cho phép mạng học được các **decision boundary phi tuyến** phức tạp — điều cần thiết để phân loại chính xác ảnh quần áo.

---
## Tổng kết & tự luận cuối bài

### 1. Một deep learning framework như PyTorch "trừu tượng hóa" những gì?

PyTorch trừu tượng hóa **3 việc cấp thấp** chính:
1. **Autograd** (Phần 3): tự tính gradient cho mọi tham số qua computation graph, không cần ta viết đạo hàm bằng tay.
2. **Hardware dispatch** (Phần 2): tự gửi phép toán sang GPU, quản lý bộ nhớ CUDA, ta chỉ cần `.to(device)`.
3. **High-level API** (Phần 4): `nn.Module`, `DataLoader`, `transforms` giúp ta mô tả kiến trúc và pipeline dữ liệu mà không cần code vòng lặp cấp thấp.

Nhờ đó ta tập trung vào **thiết kế kiến trúc model** và **thí nghiệm hyperparameter** thay vì lo viết backprop hay CUDA kernel.

### 2. Khác biệt PyTorch vs TensorFlow/Keras

| | PyTorch | TensorFlow/Keras |
|---|---|---|
| Computation graph | **Dynamic** (define-by-run) | **Static** (define-then-run, TF1) / Eager (TF2) |
| Phong cách | Pythonic, imperative | Declarative (Keras), có thể eager |
| Debug | Dễ — dùng print, pdb ngay trong forward | Khó hơn với graph mode |

**Tính "define-by-run" trong bài lab**: Ở Phần 3, mỗi lần ta viết `y = x ** 2` rồi `y.backward()`, computation graph được **tạo mới ngay lúc chạy** (không cần khai báo trước). Nếu ta đổi `y = x ** 3`, graph tự động thay đổi — đó chính là define-by-run.

### 3. Khi nào dùng gì?

| Lệnh | Khi nào | Tóm tắt 1 câu |
|---|---|---|
| `.to(device)` | Trước khi tính toán | Chuyển tensor/model sang device (GPU/CPU) để tất cả ở cùng nơi |
| `zero_grad()` | Đầu mỗi training iteration | Reset gradient về 0 để tránh tích lũy từ batch trước |
| `no_grad()` | Khi inference/evaluate | Tắt autograd để tiết kiệm bộ nhớ và tăng tốc (không cần gradient) |
| `model.eval()` | Khi inference/evaluate | Chuyển model sang chế độ đánh giá (tắt Dropout, đổi BatchNorm sang running stats) |

### 4. Vì sao deep learning cần GPU?

Dựa trên thí nghiệm speedup ở Phần 2: GPU nhanh hơn CPU ~20–50× cho phép nhân ma trận 1000×1000. Deep learning bản chất là **hàng triệu phép nhân ma trận** (forward pass qua mỗi layer = nhân ma trận, backward pass cũng vậy), lặp lại qua hàng nghìn batch và hàng chục epoch. 

Nếu giải thích cho bạn chưa học: "Hãy tưởng tượng bạn cần tính 1 triệu bài toán cộng. CPU giống 1 người giỏi toán — làm từng bài rất nhanh. GPU giống 1000 học sinh trung bình — mỗi người làm chậm hơn, nhưng cả lớp chia nhau 1 triệu bài thì xong nhanh hơn nhiều. Deep learning toàn phép nhân ma trận — chia được cho nhiều người, nên GPU thắng."

---
## Phụ lục — Đọc lỗi PyTorch thường gặp

| Thông báo lỗi | Nguyên nhân & cách xử lý |
|---|---|
| `Expected all tensors to be on the same device` | Tensor ở CPU và GPU lẫn lộn. Đưa tất cả về cùng device bằng `.to(device)`. |
| `mat1 and mat2 shapes cannot be multiplied` | Sai shape khi nhân ma trận / sai input của Linear. Đối chiếu hai con số trong ngoặc với kiến trúc model. |
| `CUDA out of memory` | Batch hoặc model quá lớn cho VRAM. Giảm `batch_size`, hoặc dùng `no_grad()` khi đánh giá. |
| `element 0 ... does not require grad` | Gọi backward() trên tensor không gắn graph, hoặc đang ở trong `no_grad()`. Kiểm tra `requires_grad`. |
| `Trying to backward through the graph a second time` | Đồ thị đã bị giải phóng. Mỗi forward chỉ backward một lần (trừ khi `retain_graph=True`). |